# Step 2.8: the full exit plan, entries, filters, and sizing

This notebook is the check of step 2.8. The execution template now runs every rule of a `StrategySpec` in both engines: an exit plan of a stop, up to three targets each closing a share of the position, breakeven, and a trailing stop; limit and stop entries; the time rules; the entry limits; and sizes worked out from money and risk (`docs/spec/fill-semantics.md` §20 to §28).

The notebook runs the example strategy of `docs/masterplan/02-engine-and-data.md` §5, with its two targets and its breakeven after the first, on one week of your 1-minute bars, through the reference engine and through the Rust engine, which must agree. It lists the trades with their exit reasons and traces one trade whose first target closed half the position and whose other half left at the stop moved to its entry price. Pick the `.venv` kernel and run all cells. The notebook reads the store in `FEFAT_DATA_DIR` and changes nothing in it.

## The product and the week

The choice of `checks/2.03.ipynb`: MES when your store holds its 1-minute bars, the contract that was the front for the most sessions in the last 120 days, and here its last five sessions.

Expected on this PC: `MES from databento: MESU2026, 5 sessions from 2026-09-08 to 2026-09-14`, or the week your store ends with.

In [ ]:
import os
from datetime import timedelta

import polars as pl
from IPython.display import display

import fefat as ff
from fefat.config import resolve_paths
from fefat.market.calendar import counted
from fefat.market.saved_instruments import instrument_table
from fefat.market.schema import Timeframe
from fefat.reference.engine import ReferenceEngine
from fefat.reference.event_log import EventKind
from fefat.reference.hours import CalendarHours
from fefat.reference.ledger import trades_of
from fefat.reference.rust_batch import EngineCase, Replay
from fefat.reference.rust_run import RustEngine, compare_engines, default_binary
from fefat.reference.scenarios import Expected, TemplateScenario
from fefat.reference.settings import RunSettings
from fefat.reference.signal_rows import SignalRow
from fefat.reference.strategy import decisions_of
from fefat.reference.tape import tape_of_bars
from fefat.reference.template import TemplateStrategy
from fefat.reference.units import instrument_parameters, money_text, price_text

paths = resolve_paths(os.environ)
instruments = instrument_table(paths.data_dir)
listed = ff.market.datasets().filter(pl.col("schema") == "ohlcv-1m")
if listed.is_empty():
    raise ValueError(
        f"Expected 1-minute bars of at least one product in {paths.data_dir}, got none: "
        "import your files first (docs/data-guide.md)."
    )
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, PROVIDER, LAST = chosen["root"], chosen["provider"], chosen["last_trade_date"]
daily = ff.market.load(
    ROOT, "1d", LAST - timedelta(days=120), LAST, fidelity="1m", provider=PROVIDER
).collect()
fronts = (
    daily.group_by("contract")
    .agg(sessions=pl.len(), last=pl.col("trade_date").max())
    .sort("sessions", "last", descending=True)
)
CONTRACT = fronts.item(0, "contract")
held = daily.filter(pl.col("contract") == CONTRACT).get_column("trade_date").sort().tail(5)
START, END = held.min(), held.max()
five_minutes = ff.market.load(CONTRACT, "5m", START, END, fidelity="1m", provider=PROVIDER)
minutes = ff.market.load(CONTRACT, "1m", START, END, fidelity="1m", provider=PROVIDER).collect()
print(f"{ROOT} from {PROVIDER}: {CONTRACT}, {counted(held.len(), 'session')} from {START} to {END}")

## The example strategy

The strategy of 02 §5 as `checks/2.03.ipynb` builds it, with two lots in place of one, so that each target has a lot to close: the first target, at 1 R, closes half the position, and breakeven then moves the stop of the other half to the entry's fill price; the second target lies at 2 R. The stop is 1.5 times the ATR of the entry's bar, rounded down to the tick, and a target in R is rounded up (F-RND-01). Entries come from 08:30 to 14:30 Chicago, three a day at most, and every position is closed by 15:05.

Expected: the specification's JSON document, with `"fraction": 0.5` on the first target, `"after_target"` in the breakeven, and `"units": 2.0`.

In [ ]:
change = pl.col("close").diff()
gain, loss = change.clip(lower_bound=0), (-change).clip(lower_bound=0)
rsi = 100 - 100 / (1 + gain.ewm_mean(alpha=1 / 14) / loss.ewm_mean(alpha=1 / 14))
before = pl.col("close").shift(1)
true_range = pl.max_horizontal(
    pl.col("high") - pl.col("low"), (pl.col("high") - before).abs(), (pl.col("low") - before).abs()
)
strat = ff.SignalStrategy(
    instrument=ROOT,
    timeframe="5m",
    features={"rsi": rsi.fill_nan(None), "atr": true_range.ewm_mean(alpha=1 / 14)},
    long_entry=pl.col("rsi") < 30,
    short_entry=pl.col("rsi") > 70,
    execution=ff.Execution(
        entry=ff.Market(),
        stop=ff.Stop(atr=1.5),
        targets=[ff.Target(r=1, fraction=0.5), ff.Target(r=2)],
        breakeven=ff.Breakeven(after_target=1),
        time=ff.TimeRules(windows=[("08:30", "14:30")], flat_by="15:05", tz="America/Chicago"),
        limits=ff.Limits(max_trades_per_day=3),
        sizing=ff.Fixed(2),
    ),
)
spec = strat.to_spec()
print(spec.document_json())

## Through both engines

The reference engine runs the strategy's template on the week's 1-minute bars, deciding at each 5-minute close: the oracle. The Rust engine runs the same week through `fefat-cli run`, the binary `just build-engine` writes, from the same signal rows; the two must give the same fills, price, time, rule, flags, and cash, and the same event log, entry by entry.

Expected on this PC: the counts of fills and exits by reason, then `The Rust engine agrees with the reference engine on the week.`

In [ ]:
instrument = instruments.get(ROOT)
zone = instrument.session_calendar().hours.time_zone
sessions = instruments.sessions_of(ROOT, paths.cache_dir)
parameters = instrument_parameters(instrument)
one_minute = Timeframe.parse("1m")
tape = tape_of_bars(minutes)
hours = CalendarHours.around(sessions, tape[0].starts_at(), tape[-1].available_at())
signals = strat.signals(five_minutes).collect()
template = TemplateStrategy(spec, signals, parameters, sessions)
engine = ReferenceEngine(RunSettings.for_fidelity(one_minute), parameters, one_minute, hours)
result = engine.run(tape, decisions_of(signals), template.on_bar)
exits = [fill for fill in result.fills if fill.role != "entry"]
reasons = pl.Series("reason", [str(fill.role) for fill in exits]).value_counts(sort=True)
print(f"Fills: {len(result.fills)}; exits by reason: {dict(reasons.iter_rows())}")

rows = tuple(
    SignalRow.model_construct(
        ts_close=row["ts_close"],
        trade_date=row["trade_date"],
        contract=row["contract"],
        long_entry=bool(row["long_entry"]),
        short_entry=bool(row["short_entry"]),
        long_exit=bool(row["long_exit"]),
        short_exit=bool(row["short_exit"]),
        entry_price=None,
        stop_price=None,
        target_price=None,
        trail_ref=None,
        size_mult=None,
        wake=True,
        features={name: row[name] for name in ("rsi", "atr") if row[name] is not None},
        flags={},
    )
    for row in signals.iter_rows(named=True)
)
week = TemplateScenario.model_construct(
    id=f"the week of {CONTRACT}",
    title="the example strategy of 02 §5",
    fidelity=one_minute,
    instrument=instrument,
    settings=RunSettings.for_fidelity(one_minute),
    fees=None,
    position=None,
    tape=tape,
    funding=(),
    expected=Expected.model_construct(fills=(), marks=(), funding=()),
    rules=("F-PLN-01",),
    strategy=spec,
    signals=rows,
)
case = EngineCase(week, hours, sessions, replay=Replay.FULL)
(comparison,) = compare_engines(RustEngine(default_binary()), [case])
differences = comparison.differences()
if differences:
    print(
        f"The engines differ in {counted(len(differences), 'place')}; the first: {differences[0]}"
    )
else:
    print("The Rust engine agrees with the reference engine on the week.")

## The trades

One row per exit: a trade the first target closes in part has a row for each of its exits, each with the entry of the whole trade, the lots it closed, why it closed them (`target_1`, `target_2`, `stop`, `flat_by`, and so on), and its profit and loss before fees. Times are Chicago's.

Expected on this PC: the week's exits, a few a day at most.

In [ ]:
def local_time(instant: int) -> str:
    """An instant as the date and time of the instrument's zone."""
    utc = pl.from_epoch(pl.lit(instant), time_unit="ns").dt.replace_time_zone("UTC")
    return pl.select(utc.dt.convert_time_zone(zone).dt.strftime("%Y-%m-%d %H:%M")).item()


TRADES = trades_of(result.fills)
table = pl.DataFrame(
    {
        "entry": [local_time(trade.entry_ts) for trade in TRADES],
        "side": [str(trade.side) for trade in TRADES],
        "lots": [trade.lots for trade in TRADES],
        "entry price": [price_text(trade.entry_price) for trade in TRADES],
        "exit": [local_time(trade.exit_ts) for trade in TRADES],
        "exit price": [price_text(trade.exit_price) for trade in TRADES],
        "exit reason": [str(trade.exit_reason) for trade in TRADES],
        "profit and loss": [money_text(trade.pnl) for trade in TRADES],
    }
)
print(
    f"{counted(len(TRADES), 'exit')} of {counted(len({trade.entry_ts for trade in TRADES}), 'trade')}"
)
display(table)

## A first target and its breakeven

The first trade of the week whose first target filled and whose other lot left later at the stop: whether breakeven moved its stop to the entry's fill price, how far beyond that price the stop filled, and the log's lines about the trade. When the first target fills, breakeven moves the stop to the entry's fill price (`moved` under F-BRK-01); on bars a stop then fills a tick beyond its level, the slippage of F-STP-04, or at the price a gap opens at.

Expected on this PC: the trade's entry, its `target_1` exit, and its `stop` exit, `True` for the stop moved to the entry's fill price, the stop filled 1 tick beyond it, and the table of its fills with the `moved` line between them; or, in a week without such a trade, a line that says so.

In [ ]:
by_entry = {}
for trade in TRADES:
    by_entry.setdefault(trade.entry_ts, []).append(trade)
TRACED = next(
    (
        exits
        for exits in by_entry.values()
        if [str(trade.exit_reason) for trade in exits][:2] == ["target_1", "stop"]
    ),
    None,
)
if TRACED is None:
    print("No trade this week has its first target filled and its other lot left at the stop.")
else:
    first, rest = TRACED
    window = (first.entry_ts, rest.exit_ts)
    traced = [
        (entry, line)
        for entry, line in zip(result.log.entries, result.log.lines(), strict=True)
        if window[0] <= entry.ts <= window[1]
        and entry.kind in (EventKind.FILLED, EventKind.MOVED, EventKind.REDUCED)
    ]
    to_entry = any(
        entry.rule == "F-BRK-01" and entry.text.endswith(f" to {price_text(first.entry_price)}")
        for entry, _ in traced
    )
    beyond = abs(first.entry_price - rest.exit_price) // parameters.tick
    print(
        f"The trade entered at {local_time(first.entry_ts)}: target_1 at {local_time(first.exit_ts)}, "
        f"the stop at {local_time(rest.exit_ts)}. Breakeven moved the stop to the entry's fill "
        f"price: {to_entry}; the stop filled {counted(beyond, 'tick')} beyond it."
    )
    display(pl.DataFrame({"the trade's fills and stop moves": [line for _, line in traced]}))

## Your checks

Run `notebooks/checks/2.08.ipynb`: it runs the example strategy with two targets and breakeven on one real week and prints the trades with exit reasons; pick one trade with exit reason `target_1` and confirm the second half later exited at the stop moved to breakeven.